In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
project_path = Path(
    r"C:\Users\ivans\Desktop\All\MSC\Data_Processsing"
    r"\assignement\air_quality_analysis"
)

input_path = (
    project_path
    / "data"
    / "processed"
    / "cleaned_air_quality_data.xlsx"
)

output_path = (
    project_path
    / "data"
    / "processed"
    / "feature_engineered_air_quality_data.xlsx"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found:\n{input_path}"
    )

df = pd.read_excel(input_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
required_columns = [
    "country_name",
    "city",
    "year",
    "pm10_concentration",
    "pm25_concentration",
    "no2_concentration",
    "latitude",
    "longitude"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        "Required columns are missing: "
        f"{missing_columns}"
    )

if df.empty:
    raise ValueError("The cleaned dataset is empty.")

if df[required_columns].isnull().any().any():
    raise ValueError(
        "The cleaned dataset still contains missing values."
    )

pollutant_columns = [
    "pm10_concentration",
    "pm25_concentration",
    "no2_concentration"
]

if (df[pollutant_columns] < 0).any().any():
    raise ValueError(
        "The cleaned dataset contains negative pollutant values."
    )

print("Validation completed successfully.")
print("Missing values:", int(df[required_columns].isnull().sum().sum()))
print("Duplicated rows:", int(df.duplicated().sum()))
print("Year range:", int(df["year"].min()), "-", int(df["year"].max()))

In [ ]:
air_quality_bins = [
    -np.inf,
    10,
    25,
    35,
    55,
    np.inf
]

air_quality_labels = [
    "Good",
    "Moderate",
    "Unhealthy for Sensitive Groups",
    "Unhealthy",
    "Very Unhealthy"
]

df["air_quality_level"] = pd.cut(
    df["pm25_concentration"],
    bins=air_quality_bins,
    labels=air_quality_labels,
    include_lowest=True,
    ordered=True
)

if df["air_quality_level"].isnull().any():
    raise ValueError(
        "Some PM2.5 values could not be assigned an air-quality level."
    )

class_counts = (
    df["air_quality_level"]
    .value_counts(sort=False)
)

class_percentages = (
    df["air_quality_level"]
    .value_counts(normalize=True, sort=False)
    .mul(100)
    .round(2)
)

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages
})

print("Air Quality Level Distribution:")
print(class_distribution)

In [ ]:
df["log_pm10_concentration"] = np.log1p(
    df["pm10_concentration"]
)

df["log_no2_concentration"] = np.log1p(
    df["no2_concentration"]
)

skewness_comparison = pd.DataFrame({
    "Feature": [
        "PM10 original",
        "PM10 log transformed",
        "NO2 original",
        "NO2 log transformed"
    ],
    "Skewness": [
        df["pm10_concentration"].skew(),
        df["log_pm10_concentration"].skew(),
        df["no2_concentration"].skew(),
        df["log_no2_concentration"].skew()
    ]
})

skewness_comparison["Absolute Skewness"] = (
    skewness_comparison["Skewness"].abs()
)

print("Skewness before and after log transformation:")
print(skewness_comparison.round(3))

print(
    "\nImportant: retain the version with the lower absolute "
    "skewness during feature selection."
)

In [ ]:
# Distance from the equator
df["absolute_latitude"] = df["latitude"].abs()

# Northern or Southern Hemisphere
df["hemisphere"] = np.where(
    df["latitude"] >= 0,
    "Northern",
    "Southern"
)

# East or West of the prime meridian
df["longitude_direction"] = np.where(
    df["longitude"] >= 0,
    "East",
    "West"
)

# Broad geographical zone based on latitude
latitude_bins = [
    -np.inf,
    -60,
    -23.5,
    23.5,
    60,
    np.inf
]

latitude_labels = [
    "Far South",
    "Southern Temperate",
    "Tropical",
    "Northern Temperate",
    "Far North"
]

df["latitude_zone"] = pd.cut(
    df["latitude"],
    bins=latitude_bins,
    labels=latitude_labels,
    right=False,
    ordered=True
)

print("Geographical features created successfully.")

print("\nHemisphere distribution:")
print(df["hemisphere"].value_counts())

print("\nLatitude-zone distribution:")
print(df["latitude_zone"].value_counts(sort=False))

In [ ]:
engineered_columns = [
    "air_quality_level",
    "log_pm10_concentration",
    "log_no2_concentration",
    "absolute_latitude",
    "hemisphere",
    "longitude_direction",
    "latitude_zone"
]

print("Newly engineered columns:")
print(engineered_columns)

print("\nFinal dataset shape:")
print(df.shape)

print("\nFinal columns:")
print(df.columns.tolist())

print("\nMissing values after feature engineering:")
print(df[engineered_columns].isnull().sum())

if df[engineered_columns].isnull().any().any():
    raise ValueError(
        "Feature engineering produced missing values."
    )

numeric_engineered_columns = [
    "log_pm10_concentration",
    "log_no2_concentration",
    "absolute_latitude"
]

if not np.isfinite(
    df[numeric_engineered_columns].to_numpy()
).all():
    raise ValueError(
        "Feature engineering produced infinite numerical values."
    )

print("\nFeature engineering validation passed.")

In [ ]:
candidate_regular_features = [
    "country_name",
    "year",
    "pm10_concentration",
    "no2_concentration",
    "latitude",
    "longitude",
    "absolute_latitude",
    "hemisphere",
    "longitude_direction",
    "latitude_zone"
]

candidate_pca_features = [
    "year",
    "log_pm10_concentration",
    "no2_concentration",
    "latitude",
    "longitude",
    "absolute_latitude"
]

target_column = "air_quality_level"

print("Candidate features for non-PCA models:")
print(candidate_regular_features)

print("\nCandidate numerical features for PCA:")
print(candidate_pca_features)

print("\nTarget column:")
print(target_column)



In [ ]:
output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_excel(
    output_path,
    index=False,
    engine="openpyxl"
)

print("Feature-engineered dataset saved successfully:")
print(output_path)